# Analiza streszczeń literatury młodzieżowej


In [ ]:
## Import bibliotek

from nltk.corpus import PlaintextCorpusReader
from nltk.tokenize import word_tokenize
import pandas as pd
import string
import numpy as np
from nltk.util import ngrams
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import operator as op
import itertools as it
import nltk
import morfeusz2
import json
import os
from wordcloud import WordCloud
from matplotlib import pyplot as plt


In [ ]:

# Zdefiniowanie funkcji pomocniczych

def print_dict(obj):
    print(json.dumps(obj, indent=4, ensure_ascii=False))

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
## Utwórz korpus dokumentów

corpus_dir = "./Literature"
# nltk.pathsec.ENFORCE = False
nltk.data.path.append(corpus_dir)
nltk.download('punkt_tab')
corpus = PlaintextCorpusReader(corpus_dir, r'.*\.txt')
file_names = corpus.fileids()

In [ ]:
documents_dict = {}
for file_name in file_names:
    documents_dict[file_name] = corpus.raw(file_name)
print_dict(documents_dict)


lengths = {}
for file_name in file_names:
    lengths[file_name] = {"pre": len(word_tokenize(documents_dict[file_name]))}
    

In [ ]:
# Utowrzenie macierzy częstości
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

matrix_tf = CountVectorizer().fit_transform(documents_df_pre['content'])

sparsity_tf = (1- (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100

In [ ]:
matrices = {
    "pre": {
        "words": matrix_tf.shape[1],
        "sparsity": sparsity_tf,
    }
}

In [ ]:
## Wstępne przygotowanie dokumentów


print(json.dumps(lengths, indent=4, ensure_ascii=False))

stopwords_path = "./stopwords_pl.txt"
with open(stopwords_path, "r", encoding="utf-8") as stopwords_file:
    stopwords = stopwords_file.read().splitlines()
    stopwords = stopwords[4:]

stopwords


In [ ]:
#use morfeusz 
def lematizacja(text):
    morfeusz = morfeusz2.Morfeusz()
    segments = it.groupby(morfeusz.analyse(text), key=op.itemgetter(0, 1))
    def disamiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group)
        perp1, lemma = min(pairs)
        return lemma.split(":")[0] if lemma else None

    lemmas = [disamiguate(group) for _, group in segments]
    return " ".join(filter(str.isalpha, lemmas))

In [ ]:
for file_name in file_names:
    documents_dict[file_name] = documents_dict[file_name].lower()
    documents_dict[file_name] = "".join([char for char in documents_dict[file_name] if char not in string.punctuation])
    documents_dict[file_name] = lematizacja(documents_dict[file_name])
    documents_dict[file_name] = " ".join([word for word in word_tokenize(documents_dict[file_name], language='polish') if word not in stopwords])
display(documents_dict)
# print (json.dumps(documents_dict, indent=4, ensure_ascii=False))

for file_name in documents_dict:
    lengths[file_name]["post"] = len(word_tokenize(documents_dict[file_name]))
# print(json.dumps(lengths, indent=4, ensure_ascii=False))


lengths = pd.DataFrame.from_dict(lengths, orient='index')
lengths["change"] = lengths["post"] - lengths["pre"]
lengths["change_percent"] = lengths["change"] / lengths["pre"] * 100




In [ ]:
documents_df_post = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

cv = CountVectorizer()
matrix_tf = cv.fit_transform(documents_df_post['content'])
tv = TfidfVectorizer()
matrix_tfidf = tv.fit_transform(documents_df_post['content'])

sparsity_tf = (1- (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100

matrices["post"] = {
    "words": matrix_tf.shape[1],
    "sparsity": sparsity_tf,
}

matrices = pd.DataFrame.from_dict(matrices, orient='index')
display(matrices)

In [ ]:
# Przygotowanie katalogów wyjściowych

create_dir("./wordclouds")
create_dir("./wordclouds/pre")
create_dir("./wordclouds/post")
create_dir("./topic_models")
create_dir("./topic_models/topics")
create_dir("./topic_models/documents")
create_dir("./clustering")
create_dir("./ngrams")



In [ ]:
# Wordclouds

wordcloud = WordCloud(
    background_color="white",
    max_words=5000,
    contour_color="steelblue",
    contour_width=3
)

for index, row in documents_df_pre.iterrows():
    wordcloud.generate(row['content'])
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/pre/{index.replace('.txt', '.png')}")
    plt.close()
    
for index, row in documents_df_post.iterrows():
    wordcloud.generate(row['content'])
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/post/{index.replace('.txt', '.png')}")
    plt.close()

In [ ]:
## Analiza tematyk


def plot_top_words(model, feature_names, n_top_words, title, subplots):
    colors = ["paleturquoise", "palevilotered", "mediumpurple", "springgreen", "lightcoral", "palegoldenrod", "salmon", "khaki", "skyblue", "stalegrey"]
    fig, axes = plt.subplots(*subplots, figsize=(30, 15), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7, color=colors[topic_idx % len(colors)])
        ax.set_title(f"Topic {topic_idx + 1}", fontdict={"fontsize": 30})
        ax.tick_params(axis="both", which="major", labelsize=20)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=40)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.savefig(f"./topic_models/topics/{title.replace(' ', '_')}.png")
    plt.close()



def plot_documents(model, matrix, n_components, title, colors):
    document_topics = pd.DataFrame(model.transform(matrix))
    document_topics.columns = [f"Temat {i + 1}" for i in range(n_components)]
    document_topics.index = [file_name.replace(".txt", "") for file_name in documents_df_post.index]
    plt.figure(figsize=(7, 4))
    left = [0] * len(document_topics)
    for i, col in enumerate(document_topics.columns):
        plt.bar(document_topics.index, document_topics[col], bottom=left, color=colors[i % len(colors)], label=col)
        left = [left[j] + document_topics[col].iloc(j) for j in range(len(document_topics))]
    plt.savefig(f"./topic_models/documents/{title.replace(' ', '_')}.png")
    plt.close()
